# 00 – Datos: descarga y calidad
**Lab #2 – Derivados Financieros** · Punto 0 (rúbrica R4 – 3 %)

Objetivo: descargar los insumos y documentar fuente, fecha de descarga, limpieza, tratamiento de dividendos y coherencia temporal.

## 0.1 Descarga de precios (CSCO, ADI, PYPL, ^NDX)

In [1]:
import yfinance as yf                      # API (no oficial) de Yahoo Finance
import pandas as pd                        # manejo de tablas
from pathlib import Path                   # rutas que sirven en Windows y Linux
from datetime import date                  # para registrar la fecha de descarga

RAW = Path("../data/raw")                  # carpeta destino (relativa a notebooks/)
RAW.mkdir(parents=True, exist_ok=True)     # la crea si no existe

TICKERS = ["CSCO", "ADI", "PYPL", "^NDX"]  # 3 acciones + índice de mercado
INICIO = "2016-10-30"                      # 10 años: el máximo de la ventana permitida
FIN = "2026-10-31"                         # yfinance NO incluye 'end' → así entra el 30/10

df = yf.download(TICKERS, start=INICIO, end=FIN,
                 auto_adjust=False,        # trae Close y Adj Close por separado
                 actions=True,             # incluye dividendos y splits
                 progress=False)

df["Close"].to_csv(RAW / "close_diario.csv")          # precio de cierre sin ajustar
df["Adj Close"].to_csv(RAW / "adjclose_diario.csv")   # ajustado por dividendos y splits
df["Dividends"].to_csv(RAW / "dividendos.csv")        # dividendos pagados por fecha

c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more detai

## 0.2 Tasa libre de riesgo (^TNX al 30/10/2026)

In [2]:
tnx = yf.download("^TNX", start="2026-10-01", end=FIN,
                  auto_adjust=False, progress=False)
tnx.to_csv(RAW / "tnx.csv")                                # serie cruda de ^TNX
rf_pct = float(tnx["Close"].squeeze().loc["2026-10-30"])   # ^TNX cotiza en puntos %
print(f"^TNX al 30/10/2026: {rf_pct:.3f} % → r_f = {rf_pct/100:.5f}")

^TNX al 30/10/2026: 4.929 % → r_f = 0.04929


c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\multi.py:220: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)
c:\Users\Lion-\miniforge3\envs\lab2-derivados\Lib\site-packages\yfinance\multi.py:224: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more detai

## 0.3 Controles de calidad

In [5]:
adj = df["Adj Close"]
print("Descargado el:", date.today())
print("Rango:", adj.index.min().date(), "→", adj.index.max().date())  # debe terminar el 2026-10-30
print("Observaciones:", len(adj))
print("=" * 30)
print("NaN por activo:\n", adj.isna().sum())

Descargado el: 2026-11-01
Rango: 2016-10-31 → 2026-10-30
Observaciones: 2514
NaN por activo:
 Ticker
ADI     0
CSCO    0
PYPL    0
^NDX    0
dtype: int64


## 0.4 Limpieza y decisiones de datos
*(ventana, frecuencia, Close vs. Adj Close — se deciden en el Punto 2)*

## 0.5 Resumen de calidad de datos (para el README y el informe)
*(redactar con palabras propias)*